## 10장 심층 신경망

### 1. 데이터를 준비하고 밀집층 만들기

1.1 패션 MNIST 데이터셋 로드하기

In [171]:
# 텐서플로 라이브러리에서 keras 패키지를 임포트한다.
from tensorflow import keras

# load_data()은 패션 MINST 데이터를 로드하고 훈련세트와 테스트세트로 나누어 반환한다.
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

1.2 데이터 전처리와 훈련세트와 검증세트 나누기

In [172]:
from sklearn.model_selection import train_test_split

# 각 픽셀값(0~255)을 255로 나누어 0~1 사이의 값으로 정규화한다.
train_scaled = train_input / 255.0

# reshape()으로 원본 배열(60000, 28, 28)의 두번째, 세번째 차원을 1차원으로 합친다.
train_scaled = train_scaled.reshape(-1, 28*28)

# 전처리된 데이터를 다시 훈련세트와 검증세트로 분리한다.
train_scaled, val_sclaed, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42
)

1.3 밀집층 2개 만들기

In [173]:
# 은닉층으로 사용될 밀집층객체 dense1을 생성한다.
dense1 = keras.layers.Dense(100, activation='sigmoid', input_shape=(784,))

# 출력층으로 사용될 밀집층객체 dense2을 생성한다.
dense2 = keras.layers.Dense(10, activation='softmax')

### 2. 심층 신경망 모델 만들기

In [174]:
# 밀집층 dense1, dense2 로 심층신경망모델을 생성한다.
model = keras.Sequential([dense1, dense2])

In [175]:
# 케라스는 모델의 summary()를 호출하면 층에 대한 유용한 정보를 얻을 수 있다.
model.summary()

Model: "sequential_14"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_28 (Dense)            (None, 100)               78500     
                                                                 
 dense_29 (Dense)            (None, 10)                1010      
                                                                 
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
_________________________________________________________________


### 3. 층을 추가하는 다른 방법

3.1 Sequential 클래스의 생성자안에서 층을 추가하는 방법

In [176]:
model = keras.Sequential([
                          keras.layers.Dense(100, activation='sigmoid', input_shape=(784,), name='hidden'),
                          keras.layers.Dense(10, activation='softmax', name='output')
], name='패션 MNIST 모델')

In [177]:
model.summary() 

Model: "패션 MNIST 모델"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hidden (Dense)              (None, 100)               78500     
                                                                 
 output (Dense)              (None, 10)                1010      
                                                                 
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
_________________________________________________________________


3.2 2. add()로 층을 추가하는 방법

In [178]:
model = keras.Sequential(name='fashionMNISTmodel_2') 
# name을 한글로 하면 3.3단계에서 오류가 나기 때문에 '패션 MNIST 모델' 대신에 'fashion MNIST model'을 사용했다.
# 공백이 있어도 오류가 나서 'fahionMNISTmodel'로 이름을 다시 바꾸었다.

model.add(keras.layers.Dense(100, activation='sigmoid', input_shape=(784,), name='hidden_2'))
model.add(keras.layers.Dense(10, activation='softmax', name='output_2'))

In [179]:
model.summary()

Model: "fashionMNISTmodel_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hidden_2 (Dense)            (None, 100)               78500     
                                                                 
 output_2 (Dense)            (None, 10)                1010      
                                                                 
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
_________________________________________________________________


3.3 층을 추가해서 다시 훈련시키기

In [180]:
model.compile(loss='sparse_categorical_crossentropy', metrics='accuracy')

model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.5600 - accuracy: 0.8089
Epoch 2/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.4079 - accuracy: 0.8524
Epoch 3/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3728 - accuracy: 0.8655
Epoch 4/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3512 - accuracy: 0.8729
Epoch 5/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3348 - accuracy: 0.8786


### 4. 렐루 활성화 함수

4.1 렐루함수와 Flatten 층을 추가해서 심층신경망 모델 만들기

In [181]:
# 신경망모델 객체를 생성한다.
model = keras.Sequential(name='fashionMNISTmodel_3')

# 입력을 1차원으로 펼치는 Flatten 층을 추가한다.
model.add(keras.layers.Flatten(input_shape=(28,28)))

# add()로 은닉층과 출력층을 추가한다.
# 은닉층의 활성화함수에 렐루(relu)함수를 지정한다.
model.add(keras.layers.Dense(100, activation='relu', name='hidden_3'))
model.add(keras.layers.Dense(10, activation='softmax', name='output_3'))

In [182]:
model.summary()

Model: "fashionMNISTmodel_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_5 (Flatten)         (None, 784)               0         
                                                                 
 hidden_3 (Dense)            (None, 100)               78500     
                                                                 
 output_3 (Dense)            (None, 10)                1010      
                                                                 
Total params: 79,510
Trainable params: 79,510
Non-trainable params: 0
_________________________________________________________________


4.2 훈련 데이터 준비하기(reshape()사용 안함)

In [183]:
(train_input, train_target), (test_input, test_target) = \
keras.datasets.fashion_mnist.load_data()

train_scaled = train_input / 255.0

train_scaled, val_sclaed, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42
)

4.3 다시 모델 설정하고 훈련시키기

In [184]:
model.compile(loss='sparse_categorical_crossentropy', metrics='accuracy')
model.fit(train_scaled, train_target, epochs=5)

Epoch 1/5
1500/1500 [==============================] - 4s 3ms/step - loss: 0.5323 - accuracy: 0.8125
Epoch 2/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3947 - accuracy: 0.8567
Epoch 3/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3575 - accuracy: 0.8719
Epoch 4/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3345 - accuracy: 0.8808
Epoch 5/5
1500/1500 [==============================] - 4s 2ms/step - loss: 0.3215 - accuracy: 0.8844


4.4 검증세트로 모델 평가하기

In [185]:
model.evaluate(val_sclaed, val_target)

375/375 [==============================] - 1s 2ms/step - loss: 0.3511 - accuracy: 0.8804


[0.35107576847076416, 0.8804166913032532]